# TrialNet — Colab GPU Training
**Base model**: `Qwen2.5-Coder-7B-Instruct` (4-bit quantized)  
**Framework**: [Unsloth](https://github.com/unslothai/unsloth) + TRL SFT  
**Output**: LoRA adapter (download → drop into Mac pipeline)  

**Runtime**: Runtime → Change runtime type → **T4 GPU** (free) or A100 (Pro)

In [ ]:
# ── 1. Verify GPU ─────────────────────────────────────────────
import subprocess
out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if out.returncode != 0:
    raise RuntimeError('No GPU detected. Runtime → Change runtime type → T4 GPU')
print(out.stdout)

In [ ]:
# ── 2. Install dependencies ───────────────────────────────────
# Unsloth auto-detects CUDA version
!pip install unsloth --quiet
!pip install datasets trl peft accelerate bitsandbytes --quiet
print('Dependencies installed.')

In [ ]:
# ── 3. Load model (4-bit) ─────────────────────────────────────
from unsloth import FastLanguageModel
import torch

MODEL_ID   = 'Qwen/Qwen2.5-Coder-7B-Instruct'
MAX_SEQ    = 2048
LORA_RANK  = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_ID,
    max_seq_length  = MAX_SEQ,
    dtype           = None,   # auto-detect bf16/fp16
    load_in_4bit    = True,
)
print(f'Loaded {MODEL_ID} in 4-bit')
print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB allocated')

In [ ]:
# ── 4. Attach LoRA adapters ───────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r               = LORA_RANK,
    target_modules  = ['q_proj','k_proj','v_proj','o_proj',
                       'gate_proj','up_proj','down_proj'],
    lora_alpha      = 32,
    lora_dropout    = 0.05,
    bias            = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state    = 42,
)
model.print_trainable_parameters()

In [ ]:
# ── 5. Build training dataset ─────────────────────────────────
# Phase 1: Reasoning traces  (100 samples)
# Phase 2: Python coding     (200 samples)
# Phase 3: Tool-use SFT      (50 samples — same as Mac Phase 5)
import random, json
from datasets import load_dataset, Dataset

random.seed(42)
all_samples = []

# --- Reasoning ---
print('Loading reasoning dataset...')
ds_reason = load_dataset('nohurry/Opus-4.6-Reasoning-3000x-filtered', split='train')
ds_reason = ds_reason.filter(lambda x: x.get('thinking') and len(x['thinking']) > 50)
ds_reason = ds_reason.select(range(min(200, len(ds_reason))))
for ex in ds_reason:
    all_samples.append({
        'messages': [
            {'role': 'user',      'content': ex['problem']},
            {'role': 'assistant', 'content': f"<thinking>\n{ex['thinking']}\n</thinking>\n\n{ex['solution']}"}
        ]
    })
print(f'  {len(all_samples)} reasoning samples')

# --- Coding ---
print('Loading coding dataset...')
ds_code = load_dataset('iamtarun/python_code_instructions_18k_alpaca', split='train').select(range(300))
for ex in ds_code:
    user_msg = f"{ex.get('instruction','')}\n\n{ex.get('input','')}".strip()
    all_samples.append({
        'messages': [
            {'role': 'user',      'content': user_msg},
            {'role': 'assistant', 'content': ex.get('output','')}
        ]
    })
print(f'  Total after coding: {len(all_samples)} samples')

# --- Tool-use SFT (inline) ---
# Calculator tool: teach model to call calculator for arithmetic, NOT for code tasks
TOOL_SCHEMAS_STR = json.dumps([
    {"name": "calculator", "description": "Evaluate a mathematical expression. Use ONLY for arithmetic/algebra — NOT for writing code.",
     "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
    {"name": "python_exec", "description": "Execute Python code and return stdout. Use ONLY when asked to RUN/TEST code, NOT when asked to WRITE code.",
     "parameters": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
])

tool_examples = [
    # Calculator — arithmetic
    ('What is 127 * 43?',
     '<tool_call>{"name": "calculator", "arguments": {"expression": "127 * 43"}}</tool_call>'),
    ('What is 15% of 840?',
     '<tool_call>{"name": "calculator", "arguments": {"expression": "0.15 * 840"}}</tool_call>'),
    ('Solve for x: 3x + 7 = 25',
     '<tool_call>{"name": "calculator", "arguments": {"expression": "(25 - 7) / 3"}}</tool_call>'),
    ('What is 2 to the power of 16?',
     '<tool_call>{"name": "calculator", "arguments": {"expression": "2 ** 16"}}</tool_call>'),
    ('Calculate the area of a circle with radius 7.',
     '<tool_call>{"name": "calculator", "arguments": {"expression": "3.14159 * 7 ** 2"}}</tool_call>'),
    # python_exec — run/test only
    ('Run this code and tell me the output: print([x**2 for x in range(5)])',
     '<tool_call>{"name": "python_exec", "arguments": {"code": "print([x**2 for x in range(5)])"}}</tool_call>'),
    ('Test this: def add(a,b): return a+b; print(add(3,4))',
     '<tool_call>{"name": "python_exec", "arguments": {"code": "def add(a,b): return a+b\nprint(add(3,4))"}}</tool_call>'),
    # No-tool — code writing (CRITICAL: no tool_call here)
    ('Write a Python function fib(n) using recursion with memoization.',
     'def fib(n, memo={}):\n    if n in memo: return memo[n]\n    if n <= 1: return n\n    memo[n] = fib(n-1, memo) + fib(n-2, memo)\n    return memo[n]'),
    ('Write a Python class Stack with push, pop, peek, is_empty, size.',
     'class Stack:\n    def __init__(self): self._data = []\n    def push(self, v): self._data.append(v)\n    def pop(self):\n        if self.is_empty(): raise IndexError("pop from empty stack")\n        return self._data.pop()\n    def peek(self):\n        if self.is_empty(): raise IndexError("peek from empty stack")\n        return self._data[-1]\n    def is_empty(self): return len(self._data) == 0\n    def size(self): return len(self._data)'),
    ('Write iterative binary search bsearch(arr, target) that returns index or -1.',
     'def bsearch(arr, target):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == target: return mid\n        elif arr[mid] < target: lo = mid + 1\n        else: hi = mid - 1\n    return -1'),
    ('Fix this buggy code: def avg(xs): return sum(xs) / len(xs)',
     'def avg(xs):\n    if not xs:\n        return None\n    return sum(xs) / len(xs)'),
    ('Write is_palindrome(s: str) -> bool that ignores case, spaces, punctuation.',
     'import re\ndef is_palindrome(s: str) -> bool:\n    cleaned = re.sub(r"[^a-z0-9]", "", s.lower())\n    return cleaned == cleaned[::-1]\n\n# Test\nassert is_palindrome("A man a plan a canal Panama") == True\nassert is_palindrome("race a car") == False'),
    ('What is the time complexity of binary search and why?',
     'O(log n). Each iteration halves the search space, so worst case needs log2(n) comparisons.'),
    ('What is machine learning in one sentence?',
     'Machine learning is a field of AI where systems learn patterns from data to make predictions or decisions without being explicitly programmed.'),
    ('All cats are animals. All animals breathe. Do all cats breathe?',
     'Yes. Since all cats are animals and all animals breathe, all cats breathe. This follows by transitivity of the universal statements.'),
    ('If it rains the ground gets wet. The ground is wet. Did it definitely rain?',
     'No, not necessarily. The ground being wet is consistent with rain, but other causes (sprinklers, flooding, spilled water) could also explain it. This is the logical fallacy of affirming the consequent.'),
]

for q, a in tool_examples:
    all_samples.append({'messages': [{'role': 'user', 'content': q}, {'role': 'assistant', 'content': a}]})

random.shuffle(all_samples)

# Train/valid split 90/10
split = int(len(all_samples) * 0.9)
train_data = all_samples[:split]
valid_data = all_samples[split:]
print(f'Train: {len(train_data)}  Valid: {len(valid_data)}')

In [ ]:
# ── 6. Format with chat template ──────────────────────────────
def format_sample(sample):
    text = tokenizer.apply_chat_template(
        sample['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {'text': text}

train_ds = Dataset.from_list(train_data).map(format_sample)
valid_ds = Dataset.from_list(valid_data).map(format_sample)
print('Sample formatted:')
print(train_ds[0]['text'][:300], '...')

In [ ]:
# ── 7. Train ──────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUTPUT_ADAPTER = '/content/trialnet_coder7b_adapter'

trainer = SFTTrainer(
    model          = model,
    tokenizer      = tokenizer,
    train_dataset  = train_ds,
    eval_dataset   = valid_ds,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size  = 2,
        gradient_accumulation_steps  = 4,   # effective batch = 8
        warmup_steps                 = 20,
        num_train_epochs             = 2,
        learning_rate                = 2e-4,
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        logging_steps                = 10,
        eval_strategy                = 'steps',
        eval_steps                   = 50,
        save_strategy                = 'epoch',
        output_dir                   = OUTPUT_ADAPTER,
        optim                        = 'adamw_8bit',
        weight_decay                 = 0.01,
        lr_scheduler_type            = 'cosine',
        seed                         = 42,
        report_to                    = 'none',
    ),
)

print('Starting training...')
trainer_stats = trainer.train()
print(f'Train loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# ── 8. Save LoRA adapter ──────────────────────────────────────
model.save_pretrained(OUTPUT_ADAPTER)
tokenizer.save_pretrained(OUTPUT_ADAPTER)
print(f'Adapter saved to {OUTPUT_ADAPTER}')

import os
files = os.listdir(OUTPUT_ADAPTER)
print('Files:', files)

In [ ]:
# ── 9. Quick sanity test ──────────────────────────────────────
FastLanguageModel.for_inference(model)

test_prompts = [
    'What is 127 * 43?',
    'Write a Python function fib(n) with memoization.',
    'Fix this buggy code: def avg(xs): return sum(xs) / len(xs)',
]

for q in test_prompts:
    messages = [{'role': 'user', 'content': q}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    with __import__('torch').no_grad():
        out = model.generate(**inputs, max_new_tokens=300, temperature=0.1, do_sample=True,
                             repetition_penalty=1.15)
    decoded = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\nQ: {q}')
    print(f'A: {decoded[:500]}')
    print('-'*60)

In [ ]:
# ── 10. Zip and download adapter ─────────────────────────────
import shutil
zip_path = '/content/trialnet_coder7b_adapter.zip'
shutil.make_archive('/content/trialnet_coder7b_adapter', 'zip', OUTPUT_ADAPTER)
print(f'Created {zip_path}')

# Download via Colab files panel
from google.colab import files
files.download(zip_path)
print('Download triggered. Check browser downloads.')

## After download — use on Mac

```bash
# Unzip into mac_llm_trialnet/
unzip trialnet_coder7b_adapter.zip -d mac_llm_trialnet/mac_trialnet_coder7b_adapter

# Convert HuggingFace adapter → MLX format
cd mac_llm_trialnet
../.venv/bin/python -m mlx_lm.convert \
    --hf-path Qwen/Qwen2.5-Coder-7B-Instruct \
    --mlx-path ./qwen_coder7b_mlx \
    -q --q-bits 4

# Update 2_mac_chatbot.py:
# MODEL_ID    = "./qwen_coder7b_mlx"
# ADAPTER_DIR = "./mac_trialnet_coder7b_adapter"
```

> **Note**: HuggingFace PEFT adapter isn't directly loadable by mlx-lm.  
> Option A: Use the Colab-trained model purely on Colab for inference.  
> Option B: Re-run Mac pipeline (`1_mac_finetune.py`) with `MODEL_ID = 'Qwen/Qwen2.5-Coder-7B-Instruct'` — MLX downloads + trains natively.